In [ ]:
!pip install -q -U transformers peft trl datasets accelerate bitsandbytes wandb

In [ ]:
import os
# --- THE ULTIMATE STABILITY FIX ---
# Hide the second GPU so Hugging Face Trainer doesn't trigger DataParallel crashes
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import os
import torch
import wandb
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM, 
    AutoTokenizer, 
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

# 1. Secure Authentication via Kaggle Secrets
print("Authenticating...")
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
wandb_key = user_secrets.get_secret("WANDB_API_KEY")

login(token=hf_token)
wandb.login(key=wandb_key)

# 2. Configuration
model_id = "meta-llama/Llama-3.2-1B-Instruct"
max_seq_length = 2048

# 3. Setup 4-bit Quantization (QLoRA)
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype
)

# 4. Load Tokenizer & Model
print("Loading Model...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
# Llama 3 does not have a default pad token, so we use the EOS token
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map={"": 0}, # This automatically spreads the model across your dual T4 GPUs
    attn_implementation="eager"
)
# 5. Prepare for LoRA Training
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# 6. Prepare the Dataset (Formatting the Argilla DPO dataset for SFT)
print("Loading and Formatting Dataset...")
dataset = load_dataset("argilla/dpo-mix-7k", split="train")

def format_sft_data(example):
    # The 'chosen' column contains the perfect conversation (User -> Assistant)
    # We apply the Llama chat template so the model learns the exact chat formatting
    text = tokenizer.apply_chat_template(example["chosen"], tokenize=False)
    return {"text": text}

# We drop the other columns because SFT only needs the 'text' to learn how to talk
sft_dataset = dataset.map(format_sft_data, remove_columns=dataset.column_names)

# 7. Configure SFTTrainer
trainer = SFTTrainer(
    model=model,
    train_dataset=sft_dataset,
    processing_class=tokenizer,
    args=SFTConfig(
        dataset_text_field="text",
        max_length=max_seq_length,
        dataset_num_proc=2,
        output_dir="./FinLlama-SFT-Output",
        per_device_train_batch_size=1,   # Reduced to fit in 15GB
        gradient_accumulation_steps=16,  # Increased to maintain learning quality
        gradient_checkpointing=True,
        warmup_steps=20,
        num_train_epochs=1, # 1 Full Epoch over the 7,000 examples
        max_grad_norm=0.3,               # The mathematical speed limit
        learning_rate=2e-5,              # Lowered from 2e-4 for stability
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="paged_adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine", # Cosine makes the learning rate curve smoother
        seed=3407,
        report_to="wandb",
        run_name="finllama-1b-sft-phase1",
    ),
)

# 8. Start Training 🚀
print("Starting SFT Training Phase...")
trainer.train()

# 9. Save Locally & Push to Hugging Face Hub!
print("Saving a local backup to Kaggle disk...")
trainer.model.save_pretrained("/kaggle/working/Llama-Local-Save")
tokenizer.save_pretrained("/kaggle/working/Llama-Local-Save")

print("Pushing trained adapters to Hugging Face...")
# REPLACE 'your-username' WITH YOUR ACTUAL HUGGING FACE USERNAME
trainer.model.push_to_hub("pranav6905/Llama-3.2-1B-SFT-DPOMix-Adapters", token=hf_token)
tokenizer.push_to_hub("pranav6905/Llama-3.2-1B-SFT-DPOMix-Adapters", token=hf_token)

print("Phase 1 Complete! Check your Hugging Face profile!")